In [8]:
import nltk
import random
import nlpaug.augmenter.word as naw
import numpy as np
import pandas as pd
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
import re
import string
from typing import Dict, Any, List

In [3]:
dataset = pd.read_csv('../Data/cvs_dataset/final_cv_dataset.csv')
dataset = dataset.dropna().reset_index(drop=True)
dataset.head()

,text,token_count,char_count,avg_word_length,unique_word_ratio,stopword_ratio,punctuation_count,comma_count,period_count,digit_count,uppercase_ratio,bullet_point_flag,contains_year,special_char_count,avg_sentence_length,long_word_ratio,titlecase_ratio,label,label_full
0,Project under Graduation,4,28,6.25,1.0000,0.2500,0,0,0,0,0.1200,0,0,0,4.0,0.5000,0.7500,Exp,Experience
1,Father’s Name,4,16,3.25,1.0000,0.2500,0,0,0,0,0.3077,0,0,0,4.0,0.0000,0.5000,PI,Personal Info
2,"Omgeo Connect, a complementary offering to Omg...",114,858,6.40,0.6579,0.2544,16,5,3,0,0.0205,0,0,6,28.5,0.5175,0.1053,Exp,Experience
3,Implemented Singleton pattern for property loa...,18,117,5.39,0.9444,0.2222,3,2,1,0,0.0515,0,0,0,18.0,0.4444,0.1667,Exp,Experience
4,· Developed report for Plan Vs Actual Billing ...,33,196,4.85,0.6364,0.2727,2,0,2,0,0.0938,0,0,0,16.5,0.2121,0.2424,Exp,Experience


In [10]:
# Download required NLTK data (run once)
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpora/stopwords')
except LookupError:
    print("Downloading required NLTK data...")
    nltk.download('punkt')
    nltk.download('stopwords')

# Load English stopwords
ENGLISH_STOPWORDS = set(stopwords.words('english'))

In [9]:
def _get_default_features() -> Dict[str, Any]:
    return {
        'token_count': 0,
        'char_count': 0,
        'avg_word_length': 0.0,
        'unique_word_ratio': 0.0,
        'stopword_ratio': 0.0,
        'punctuation_count': 0,
        'comma_count': 0,
        'period_count': 0,
        'digit_count': 0,
        'uppercase_ratio': 0.0,
        'bullet_point_flag': 0,
        'contains_year': 0,
        'special_char_count': 0,
        'avg_sentence_length': 0.0,
        'long_word_ratio': 0.0,
        'titlecase_ratio': 0.0
    }

In [11]:
def _compute_writing_style_features(text: str) -> dict:
    # Handle None or empty text
    if pd.isna(text) or not isinstance(text, str) or not text.strip():
        return _get_default_features()
    
    text = str(text).strip()
    
    # Basic character and word analysis
    char_count = len(text)
    
    # Tokenize into words
    try:
        tokens = word_tokenize(text.lower())
        word_tokens = [token for token in tokens if any(c.isalnum() for c in token)]
        token_count = len(word_tokens)
    except:
        word_tokens = [word for word in text.lower().split() if any(c.isalnum() for c in word)]
        token_count = len(word_tokens)
    
    # Average word length
    if word_tokens:
        clean_tokens = [re.sub(r'[^\w]', '', token) for token in word_tokens if token]
        clean_tokens = [token for token in clean_tokens if token]
        avg_word_length = sum(len(token) for token in clean_tokens) / len(clean_tokens) if clean_tokens else 0
    else:
        avg_word_length = 0
    
    # Unique word ratio
    unique_word_ratio = len(set(word_tokens)) / token_count if token_count > 0 else 0
    
    # Stopword ratio
    stopword_count = sum(1 for token in word_tokens if token.lower() in ENGLISH_STOPWORDS)
    stopword_ratio = stopword_count / token_count if token_count > 0 else 0
    
    # Punctuation analysis
    punctuation_count = sum(1 for char in text if char in string.punctuation)
    comma_count = text.count(',')
    period_count = text.count('.')
    
    # Digit count
    digit_count = sum(1 for char in text if char.isdigit())
    
    # Uppercase ratio
    alpha_chars = [char for char in text if char.isalpha()]
    uppercase_ratio = sum(1 for char in alpha_chars if char.isupper()) / len(alpha_chars) if alpha_chars else 0
    
    # Bullet point detection
    bullet_patterns = [r'^\s*[-•*]', r'^\s*\d+[\.\)]', r'^\s*[a-zA-Z][\.\)]']
    bullet_point_flag = any(re.match(pattern, text) for pattern in bullet_patterns)
    
    # Year detection
    year_pattern = r'\b(19|20)\d{2}\b'
    contains_year = bool(re.search(year_pattern, text))
    
    # Special character count
    special_chars = set('@/-')
    special_char_count = sum(1 for char in text if char in special_chars)
    
    # Sentence-level analysis
    try:
        sentences = sent_tokenize(text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        if sentences:
            sentence_word_counts = []
            for sentence in sentences:
                sentence_tokens = word_tokenize(sentence.lower())
                sentence_word_count = len([t for t in sentence_tokens if any(c.isalnum() for c in t)])
                sentence_word_counts.append(sentence_word_count)
            avg_sentence_length = sum(sentence_word_counts) / len(sentence_word_counts)
        else:
            avg_sentence_length = 0
    except:
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        avg_sentence_length = sum(len(s.split()) for s in sentences) / len(sentences) if sentences else 0
    
    # Long word ratio
    if word_tokens:
        clean_tokens = [re.sub(r'[^\w]', '', token) for token in word_tokens if token]
        clean_tokens = [token for token in clean_tokens if token]
        long_words = [token for token in clean_tokens if len(token) > 6]
        long_word_ratio = len(long_words) / len(clean_tokens) if clean_tokens else 0
    else:
        long_word_ratio = 0
    
    # Title case ratio
    if word_tokens:
        original_tokens = word_tokenize(text)
        word_only_tokens = [token for token in original_tokens if any(c.isalpha() for c in token)]
        titlecase_ratio = len([token for token in word_only_tokens if token.istitle()]) / len(word_only_tokens) if word_only_tokens else 0
    else:
        titlecase_ratio = 0
    
    return {
        'token_count': token_count,
        'char_count': char_count,
        'avg_word_length': round(avg_word_length, 2),
        'unique_word_ratio': round(unique_word_ratio, 4),
        'stopword_ratio': round(stopword_ratio, 4),
        'punctuation_count': punctuation_count,
        'comma_count': comma_count,
        'period_count': period_count,
        'digit_count': digit_count,
        'uppercase_ratio': round(uppercase_ratio, 4),
        'bullet_point_flag': int(bullet_point_flag),
        'contains_year': int(contains_year),
        'special_char_count': special_char_count,
        'avg_sentence_length': round(avg_sentence_length, 2),
        'long_word_ratio': round(long_word_ratio, 4),
        'titlecase_ratio': round(titlecase_ratio, 4)
    }

In [14]:
def balance_classes_df(df, text_col='text', label_col='label_full', verbose=True):

    sentences = df[text_col].tolist()
    labels = df[label_col].tolist()
    
    sentences = np.array(sentences)
    labels = np.array(labels)
    unique_labels, counts = np.unique(labels, return_counts=True)
    max_count = max(counts)
    majority_label = unique_labels[np.argmax(counts)]
    
    if verbose:
        print(f"Majority class '{majority_label}' has {max_count} samples. Targeting all classes to this size.")
    
    # Set up augmenters
    augmenters = [
        naw.SynonymAug(aug_src='wordnet', aug_max=2),
        naw.AntonymAug(aug_max=1),
        naw.RandomWordAug(action='delete', aug_max=2, aug_p=0.3)
    ]
    
    augmented_df = df.copy()
    
    for lbl in unique_labels:
        count = (augmented_df[label_col] == lbl).sum()
        if count < max_count:
            if verbose:
                print(f"Augmenting '{lbl}': {count} -> {max_count} (need {max_count - count} more)")
            
            class_mask = labels == lbl
            class_sentences = sentences[class_mask]
            class_indices = np.where(class_mask)[0]
            
            num_to_augment = max_count - count
            to_augment_indices = np.random.choice(len(class_sentences), num_to_augment, replace=True)
            to_augment = class_sentences[to_augment_indices]
            
            new_rows = []
            for i, orig in enumerate(to_augment):
                # Random augmenter with synonym bias
                probs = [0.6, 0.2, 0.2]  # 60% synonym for safety
                chosen_idx = np.random.choice(len(augmenters), p=probs)
                aug_sent = augmenters[chosen_idx].augment([orig])[0]
                
                # Compute features for augmented text
                features = _compute_writing_style_features(aug_sent)
                features[text_col] = aug_sent
                features[label_col] = lbl
                
                # Add other original columns (e.g., 'label') with their original values
                for col in df.columns:
                    if col not in [text_col, label_col] + list(features.keys()):
                        features[col] = df.loc[class_indices[0], col]  # Use first row as default for non-feature cols
                    
                new_rows.append(features)
                
                if verbose and (i + 1) % 1000 == 0:
                    print(f"  ... Generated {i + 1}/{num_to_augment} for '{lbl}'")
            
            # Append new rows
            new_df = pd.DataFrame(new_rows)
            augmented_df = pd.concat([augmented_df, new_df], ignore_index=True)
            
            if verbose:
                print(f"  Done: '{lbl}' now at {max_count}")
    
    if verbose:
        print(f"\nFinal counts: {augmented_df[label_col].value_counts().to_dict()}")
        print(f"Total dataset size: {len(augmented_df)} (grew by {len(augmented_df) - len(df)})")
    
    return augmented_df

In [15]:
augmented_dataset = balance_classes_df(dataset)
augmented_dataset.head()

Majority class 'Experience' has 41158 samples. Targeting all classes to this size.
Augmenting 'Education': 9494 -> 41158 (need 31664 more)
  ... Generated 1000/31664 for 'Education'
  ... Generated 2000/31664 for 'Education'
  ... Generated 3000/31664 for 'Education'
  ... Generated 4000/31664 for 'Education'
  ... Generated 5000/31664 for 'Education'
  ... Generated 6000/31664 for 'Education'
  ... Generated 7000/31664 for 'Education'
  ... Generated 8000/31664 for 'Education'
  ... Generated 9000/31664 for 'Education'
  ... Generated 10000/31664 for 'Education'
  ... Generated 11000/31664 for 'Education'
  ... Generated 12000/31664 for 'Education'
  ... Generated 13000/31664 for 'Education'
  ... Generated 14000/31664 for 'Education'
  ... Generated 15000/31664 for 'Education'
  ... Generated 16000/31664 for 'Education'
  ... Generated 17000/31664 for 'Education'
  ... Generated 18000/31664 for 'Education'
  ... Generated 19000/31664 for 'Education'
  ... Generated 20000/31664 for 'E

,text,token_count,char_count,avg_word_length,unique_word_ratio,stopword_ratio,punctuation_count,comma_count,period_count,digit_count,uppercase_ratio,bullet_point_flag,contains_year,special_char_count,avg_sentence_length,long_word_ratio,titlecase_ratio,label,label_full
0,Project under Graduation,4,28,6.25,1.0000,0.2500,0,0,0,0,0.1200,0,0,0,4.0,0.5000,0.7500,Exp,Experience
1,Father’s Name,4,16,3.25,1.0000,0.2500,0,0,0,0,0.3077,0,0,0,4.0,0.0000,0.5000,PI,Personal Info
2,"Omgeo Connect, a complementary offering to Omg...",114,858,6.40,0.6579,0.2544,16,5,3,0,0.0205,0,0,6,28.5,0.5175,0.1053,Exp,Experience
3,Implemented Singleton pattern for property loa...,18,117,5.39,0.9444,0.2222,3,2,1,0,0.0515,0,0,0,18.0,0.4444,0.1667,Exp,Experience
4,· Developed report for Plan Vs Actual Billing ...,33,196,4.85,0.6364,0.2727,2,0,2,0,0.0938,0,0,0,16.5,0.2121,0.2424,Exp,Experience


In [16]:
augmented_dataset.to_csv("../Data/cvs_dataset/final_augmented_dataset.csv", index=False)